# AMI: Average Words per Topic (Ground Truth)

This notebook computes **per-topic word counts** and the **average words per topic** from AMI **manual** `*.topic.xml` files.

AMI manual distribution laid out like:
- `dataset/ami_public_manual_1.6.2/words/<MEETING>.*.words.xml`
- `dataset/ami_public_manual_1.6.2/topics/<MEETING>.topic.xml`


In [8]:
from lxml import etree
from operator import itemgetter
import glob
import os
import json
from pathlib import Path

# ---- Configure paths ----
DATA_ROOT = Path('/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/ami_public_manual_1.6.2')
WORDS_DIR = DATA_ROOT / 'words'
TOPICS_DIR = DATA_ROOT / 'topics'
OUT_DIR = Path('/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/topic_stats')
OUT_DIR.mkdir(parents=True, exist_ok=True)

NS = {'nite': 'http://nite.sourceforge.net/'}

print('WORDS_DIR:', WORDS_DIR)
print('TOPICS_DIR:', TOPICS_DIR)
print('OUT_DIR:', OUT_DIR)

WORDS_DIR: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/ami_public_manual_1.6.2/words
TOPICS_DIR: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/ami_public_manual_1.6.2/topics
OUT_DIR: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/topic_stats


## Parse words.xml (manual)

We keep each token with:
- `id`, `start`, `end`, `speaker`, `text`
- `word_type` (`w` for words; silences etc. are other tags)
- `punctuation` (from `punc` attribute when present)


In [ ]:
def manual_parse_words_and_silences(words_path):
    tree = etree.parse(str(words_path))
    root = tree.getroot()

    words = []
    for word_elem in root:
        word_id = word_elem.get('{http://nite.sourceforge.net/}id')

        # If missing times, fall back safely
        if word_elem.get('starttime') is None:
            start = words[-1]['end'] if words else 0.0
        else:
            start = float(word_elem.get('starttime'))

        if word_elem.get('endtime') is None:
            end = start
        else:
            end = float(word_elem.get('endtime'))

        speaker = word_elem.get('speaker')
        text = word_elem.text.strip() if word_elem.text else ''

        words.append({
            'id': word_id,
            'start': start,
            'end': end,
            'speaker': speaker,
            'text': text,
            'punctuation': word_elem.get('punc', ''),
            'word_type': word_elem.tag,
        })
    return words


def combine_words_from_multiple_files(file_paths):
    """Combine words from multiple participants' words.xml into one time-sorted list."""
    all_words = []
    all_words_dict = {}

    for file_path in file_paths:
        words = manual_parse_words_and_silences(file_path)
        all_words.extend(words)
        for w in words:
            wid = w['id']
            if wid not in all_words_dict:
                all_words_dict[wid] = w
            else:
                # Merge duplicate IDs if they occur
                existing = all_words_dict[wid]
                existing['text'] = (existing.get('text', '') + ' ' + w.get('text', '')).strip()
                existing['end'] = max(existing.get('end', 0.0), w.get('end', 0.0))
                existing['punctuation'] = existing.get('punctuation', '') or w.get('punctuation', '')
                existing['word_type'] = existing.get('word_type', '') or w.get('word_type', '')

    all_words_sorted = sorted(all_words, key=itemgetter('start'))
    return all_words_sorted, all_words_dict

## Parse ground-truth topics (`*.topic.xml`)

This version correctly aggregates all `<nite:child>` spans per topic and takes the **min start** and **max end** boundaries.

In [ ]:
def parse_topics(topic_path, word_times):
    tree = etree.parse(str(topic_path))
    root = tree.getroot()

    topics = []
    for topic in root.findall('.//topic'):
        label = topic.get('other_description', 'Unknown').strip()

        start_time_list, end_time_list = [], []
        start_idx_list, end_idx_list = [], []

        childs = topic.findall('.//nite:child', namespaces=NS)
        for child in childs:
            href = child.get('href')
            if not href or '#' not in href:
                continue

            _, fragment = href.split('#', 1)
            parts = fragment.replace('id(', '').replace(')', '').split('..')
            start_id = parts[0]
            end_id = parts[1] if len(parts) == 2 else start_id

            index_start = next((i for i, w in enumerate(word_times) if w['id'] == start_id), -1)
            index_end = next((i for i, w in enumerate(word_times) if w['id'] == end_id), -1)

            if index_start == -1 or index_end == -1:
                continue

            start_time_list.append(word_times[index_start]['start'])
            end_time_list.append(word_times[index_end]['end'])
            start_idx_list.append(index_start)
            end_idx_list.append(index_end)

        if not start_idx_list:
            continue

        topics.append({
            'label': label,
            'start_word': min(start_idx_list),
            'end_word': max(end_idx_list),
            'start_time': min(start_time_list),
            'end_time': max(end_time_list),
        })

    return topics

In [23]:
def topic_word_stats(topics_ref, word_times):
    per_topic = []

    for t in topics_ref:
        s = t["start_word"]
        e = t["end_word"]

        if s < 0 or e < 0 or s > e:
            continue

        word_count = sum(
            1 for w in word_times[s:e+1]
            if w.get("word_type") == "w"
        )

        per_topic.append({
            "label": t["label"],
            "start_time": t["start_time"],
            "end_time": t["end_time"],
            "start_word": s,
            "end_word": e,
            "word_count": word_count
        })

    avg_words = sum(t["word_count"] for t in per_topic) / len(per_topic) if per_topic else 0.0
    return per_topic, avg_words


## Compute per-topic word counts + average

Counts **only** `word_type == 'w'` to exclude silences and other non-word tags.

In [ ]:
def topic_word_stats(topics_ref, word_times):
    per_topic = []
    for t in topics_ref:
        s = t['start_word']
        e = t['end_word']
        if s is None or e is None or s < 0 or e < 0 or s > e:
            continue

        wc = sum(1 for w in word_times[s:e+1] if w.get('word_type') == 'w')

        per_topic.append({
            'label': t['label'],
            'start_time': t['start_time'],
            'end_time': t['end_time'],
            'start_word': s,
            'end_word': e,
            'word_count': wc,
        })

    avg_words = (sum(x['word_count'] for x in per_topic) / len(per_topic)) if per_topic else 0.0
    return per_topic, avg_words

## Run for one meeting

Set `MEETING_ID` (e.g., `EN2001a`).

In [ ]:
MEETING_ID = 'ES2002a'  # <- change me

word_file_paths = glob.glob(str(WORDS_DIR / f"{MEETING_ID}.*.words.xml"))
topic_xml = TOPICS_DIR / f"{MEETING_ID}.topic.xml"

if not word_file_paths:
    raise FileNotFoundError(f"No words.xml found for {MEETING_ID} in {WORDS_DIR}")
if not topic_xml.exists():
    raise FileNotFoundError(f"No topic.xml found for {MEETING_ID} at {topic_xml}")

combined_words, combined_words_dic = combine_words_from_multiple_files(word_file_paths)
topics_ref = parse_topics(topic_xml, combined_words)
per_topic, avg_words = topic_word_stats(topics_ref, combined_words)

print(f"[{MEETING_ID}] topics={len(per_topic)} avg_words_per_topic={avg_words:.2f}")
per_topic[:3]

[ES2002a] topics=7 avg_words_per_topic=443.57


[{'label': 'introduction of participants and their roles',
  'start_time': 50.42,
  'end_time': 88.71,
  'start_word': 0,
  'end_word': 108,
  'word_count': 106},
 {'label': 'project goals and design process',
  'start_time': 89.3,
  'end_time': 151.79,
  'start_word': 109,
  'end_word': 310,
  'word_count': 198},
 {'label': 'drawing animals on the whiteboard',
  'start_time': 151.79,
  'end_time': 453.61,
  'start_word': 311,
  'end_word': 1075,
  'word_count': 703}]

## Run for all meetings + corpus average

This iterates over all `*.topic.xml` files and computes a corpus-level average words/topic.

In [ ]:
TOTAL_TOPIC_WORDS = 0
TOTAL_TOPIC_COUNT = 0

results = []

used_files = os.listdir("/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/amicorpus")

topic_files = sorted(TOPICS_DIR.glob('*.topic.xml'))
print('Found topic files:', len(topic_files))

for tf in topic_files:
    if tf.name.split('.')[0] not in used_files:
        print(f"Skipping {tf.name} as it's not in used_files")
        continue
    
    meeting_id = tf.stem.split('.')[0]  # handles cases like EN2001a.topic.xml

    word_file_paths = glob.glob(str(WORDS_DIR / f"{meeting_id}.*.words.xml"))
    if not word_file_paths:
        continue

    combined_words, _ = combine_words_from_multiple_files(word_file_paths)
    topics_ref = parse_topics(tf, combined_words)
    per_topic, avg_words = topic_word_stats(topics_ref, combined_words)

    if not per_topic:
        continue

    TOTAL_TOPIC_WORDS += sum(t['word_count'] for t in per_topic)
    TOTAL_TOPIC_COUNT += len(per_topic)

    results.append({
        'meeting_id': meeting_id,
        'num_topics': len(per_topic),
        'avg_words_per_topic': avg_words,
    })

corpus_avg = (TOTAL_TOPIC_WORDS / TOTAL_TOPIC_COUNT) if TOTAL_TOPIC_COUNT else 0.0
print(f"[AMI CORPUS] meetings={len(results)} topics={TOTAL_TOPIC_COUNT} avg_words_per_topic={corpus_avg:.2f}")

# Save summary
with open(OUT_DIR / 'ami_corpus_topic_word_summary.json', 'w') as f:
    json.dump({'corpus_avg_words_per_topic': corpus_avg, 'meetings': results}, f, indent=2)
    print(len(results))


Found topic files: 139
Skipping ES2002b.topic.xml as it's not in used_files
Skipping ES2002c.topic.xml as it's not in used_files
Skipping ES2002d.topic.xml as it's not in used_files
Skipping ES2003b.topic.xml as it's not in used_files
Skipping ES2003d.topic.xml as it's not in used_files
Skipping ES2004a.topic.xml as it's not in used_files
Skipping ES2004b.topic.xml as it's not in used_files
Skipping ES2004c.topic.xml as it's not in used_files
Skipping ES2004d.topic.xml as it's not in used_files
Skipping ES2005a.topic.xml as it's not in used_files
Skipping ES2005b.topic.xml as it's not in used_files
Skipping ES2005c.topic.xml as it's not in used_files
Skipping ES2005d.topic.xml as it's not in used_files
Skipping ES2006a.topic.xml as it's not in used_files
Skipping ES2006b.topic.xml as it's not in used_files
Skipping ES2006d.topic.xml as it's not in used_files
Skipping ES2007a.topic.xml as it's not in used_files
Skipping ES2007b.topic.xml as it's not in used_files
Skipping ES2007c.topic.

# AMI: Average Words per Utterance (Ground Truth)

This notebook computes **per-topic word counts** and the **average words per topic** from AMI **manual** `*.topic.xml` files.

AMI manual distribution laid out like:
- `dataset/ami_public_manual_1.6.2/words/<MEETING>.*.words.xml`
- `dataset/ami_public_manual_1.6.2/topics/<MEETING>.topic.xml`


In [41]:
from lxml import etree
from operator import itemgetter
import glob
import os
import json
from pathlib import Path

# ---- Configure paths ----
DATA_ROOT = Path('/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/ami_public_manual_1.6.2')
WORDS_DIR = DATA_ROOT / 'words'
TOPICS_DIR = DATA_ROOT / 'topics'
OUT_DIR = Path('/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/topic_stats')
OUT_DIR.mkdir(parents=True, exist_ok=True)

NS = {'nite': 'http://nite.sourceforge.net/'}

print('WORDS_DIR:', WORDS_DIR)
print('TOPICS_DIR:', TOPICS_DIR)
print('OUT_DIR:', OUT_DIR)

WORDS_DIR: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/ami_public_manual_1.6.2/words
TOPICS_DIR: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/ami_public_manual_1.6.2/topics
OUT_DIR: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/topic_stats


In [31]:
def parse_manual_segments(segments_path):
    tree = etree.parse(str(segments_path))
    root = tree.getroot()

    segments = []
    for seg in root.findall('segment'):
        segment_id = seg.get('nite:id')
        start_time = float(seg.get('transcriber_start'))
        end_time = float(seg.get('transcriber_end'))

        child = seg.find('nite:child', namespaces=NS)
        if child is None:
            continue

        href = child.get('href')
        _, fragment = href.split('#')

        parts = fragment.replace('id(', '').replace(')', '').split('..')
        start_id = parts[0]
        end_id = parts[1] if len(parts) == 2 else start_id

        segments.append({
            "segment_id": segment_id,
            "start_id": start_id,
            "end_id": end_id,
            "start_time": start_time,
            "end_time": end_time,
        })

    return segments


In [32]:
def combine_segments_from_multiple_files(segment_files):
    all_segments = []
    for sf in segment_files:
        all_segments.extend(parse_manual_segments(sf))
    return sorted(all_segments, key=lambda x: x["start_time"])


In [33]:
def utterance_word_stats(segments, word_dict):
    utterances = []

    word_ids = list(word_dict.keys())

    for seg in segments:
        try:
            s_idx = word_ids.index(seg["start_id"])
            e_idx = word_ids.index(seg["end_id"]) + 1
        except ValueError:
            continue

        words = list(word_dict.values())[s_idx:e_idx]
        tokens = [w["text"] for w in words if w.get("word_type") == "w"]

        utterances.append({
            "segment_id": seg["segment_id"],
            "start_time": seg["start_time"],
            "end_time": seg["end_time"],
            "word_count": len(tokens),
            "text": " ".join(tokens)
        })

    avg_words = sum(u["word_count"] for u in utterances) / len(utterances) if utterances else 0.0
    return utterances, avg_words


In [ ]:
TOTAL_WORDS = 0
TOTAL_UTTERANCES = 0

used_files = os.listdir("/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/amicorpus")

topic_files = sorted(TOPICS_DIR.glob('*.topic.xml'))

for tf in TOPICS_DIR.glob("*.topic.xml"):
    if tf.name.split(".")[0] not in used_files:
        print(f"Skipping {tf.name} as it's not in used_files")
        continue
    meeting_id = tf.stem.split(".")[0]

    word_files = glob.glob(str(WORDS_DIR / f"{meeting_id}.*.words.xml"))
    segment_files = glob.glob(str(DATA_ROOT / "segments" / f"{meeting_id}.*.segments.xml"))

    if not word_files or not segment_files:
        continue

    combined_words, combined_words_dict = combine_words_from_multiple_files(word_files)
    segments = combine_segments_from_multiple_files(segment_files)
    utterances, _ = utterance_word_stats(segments, combined_words_dict)

    TOTAL_WORDS += sum(u["word_count"] for u in utterances)
    TOTAL_UTTERANCES += len(utterances)

corpus_avg = TOTAL_WORDS / TOTAL_UTTERANCES if TOTAL_UTTERANCES else 0.0
print(f"[AMI CORPUS] utterances={TOTAL_UTTERANCES} avg_words_per_utterance={corpus_avg:.2f}")



Found topic files: 139
Skipping TS3008a.topic.xml as it's not in used_files
Skipping TS3003b.topic.xml as it's not in used_files
Skipping ES2010d.topic.xml as it's not in used_files
Skipping IS1006a.topic.xml as it's not in used_files
Skipping ES2004b.topic.xml as it's not in used_files
Skipping IS1002c.topic.xml as it's not in used_files
Skipping TS3005a.topic.xml as it's not in used_files
Skipping TS3010c.topic.xml as it's not in used_files
Skipping TS3006d.topic.xml as it's not in used_files
Skipping TS3012b.topic.xml as it's not in used_files
Skipping IS1000b.topic.xml as it's not in used_files
Skipping ES2009b.topic.xml as it's not in used_files
Skipping ES2004d.topic.xml as it's not in used_files
Skipping ES2010b.topic.xml as it's not in used_files
Skipping ES2007a.topic.xml as it's not in used_files
Skipping TS3009c.topic.xml as it's not in used_files
Skipping TS3003d.topic.xml as it's not in used_files
Skipping ES2012c.topic.xml as it's not in used_files
Skipping TS3012d.topic.

# AMI: Average tokens


In [9]:
import os
import glob
import json
from pathlib import Path
from operator import itemgetter
from lxml import etree
from llama3_tokenizer import Tokenizer

# ---------------------------
# 1) Llama3 Tokenizer classes
# ---------------------------
# Paste your Tokenizer + ChatFormat code ABOVE this block (exactly as you have it).
# (Do not paste again here.)

# ---------------------------
# 2) Paths / constants
# ---------------------------

SEGMENTS_DIR = DATA_ROOT / "segments"
TOKENIZER_MODEL_PATH = "/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/Meta-Llama-3-8B-Instruct/original/tokenizer.model"  # <-- CHANGE THIS


# ---------------------------
# 3) Load tokenizer
# ---------------------------
tokenizer = Tokenizer(TOKENIZER_MODEL_PATH)

def count_llama_tokens(text: str) -> int:
    # No BOS/EOS for length; you can set bos/eos True if you want model-input length.
    return len(tokenizer.encode(text, bos=False, eos=False))


# ---------------------------
# 4) Words parsing (manual)
# ---------------------------
def manual_parse_words_and_silences(words_path):
    tree = etree.parse(str(words_path))
    root = tree.getroot()

    words = []
    for word_elem in root:
        word_id = word_elem.get('{http://nite.sourceforge.net/}id')

        # Safe fallback if missing times
        if word_elem.get('starttime') is None:
            start = words[-1]['end'] if words else 0.0
        else:
            start = float(word_elem.get('starttime'))

        if word_elem.get('endtime') is None:
            end = start
        else:
            end = float(word_elem.get('endtime'))

        speaker = word_elem.get('speaker')
        text = word_elem.text.strip() if word_elem.text else ''

        words.append({
            'id': word_id,
            'start': start,
            'end': end,
            'speaker': speaker,
            'text': text,
            'punctuation': word_elem.get('punc', ''),
            'word_type': word_elem.tag,  # 'w' for words
        })
    return words


def combine_words_from_multiple_files(file_paths):
    """
    Returns:
      - all_words_sorted: list of word dicts sorted by time
      - all_words_dict: dict keyed by word_id (first-seen version)
      - id2idx: map from word_id -> index in all_words_sorted (fast lookup)
    """
    all_words = []
    all_words_dict = {}

    for fp in file_paths:
        words = manual_parse_words_and_silences(fp)
        all_words.extend(words)
        for w in words:
            wid = w["id"]
            if wid not in all_words_dict:
                all_words_dict[wid] = w

    all_words_sorted = sorted(all_words, key=itemgetter("start"))

    id2idx = {}
    for i, w in enumerate(all_words_sorted):
        # if duplicates exist, keep earliest index
        if w["id"] not in id2idx:
            id2idx[w["id"]] = i

    return all_words_sorted, all_words_dict, id2idx


def words_span_text(word_times, start_idx: int, end_idx: int) -> str:
    """
    Build a clean text from word_times[start_idx:end_idx+1].
    Uses punctuation like your transcript builder.
    """
    out = ""
    for w in word_times[start_idx:end_idx+1]:
        if w.get("word_type") != "w":
            continue
        token = w.get("text", "")
        if not token:
            continue
        if w.get("punctuation"):
            out = out[:-1] + token + " " if out else token + " "
        else:
            out += token + " "
    return out.strip()


# ---------------------------
# 5) Utterance parsing (segments.xml)
# ---------------------------
def parse_manual_segments(segments_path):
    tree = etree.parse(str(segments_path))
    root = tree.getroot()

    segments = []
    for seg in root.findall('segment'):
        segment_id = seg.get('nite:id')
        start_time = seg.get('transcriber_start')
        end_time = seg.get('transcriber_end')

        child = seg.find('nite:child', namespaces=NS)
        if child is None:
            continue

        href = child.get('href')
        if not href or "#" not in href:
            continue

        _, fragment = href.split('#', 1)
        parts = fragment.replace('id(', '').replace(')', '').split('..')
        start_id = parts[0]
        end_id = parts[1] if len(parts) == 2 else start_id

        segments.append({
            'segment_id': segment_id,
            'start_id': start_id,
            'end_id': end_id,
            'start_time': float(start_time) if start_time is not None else None,
            'end_time': float(end_time) if end_time is not None else None,
        })
    return segments


def combine_segments_from_multiple_files(segment_files):
    all_segments = []
    for sf in segment_files:
        all_segments.extend(parse_manual_segments(sf))
    return sorted(all_segments, key=lambda x: (x["start_time"] is None, x["start_time"]))


def utterance_token_stats(segments, word_times, id2idx):
    utterances = []
    for seg in segments:
        s_id, e_id = seg["start_id"], seg["end_id"]
        if s_id not in id2idx or e_id not in id2idx:
            continue

        s_idx = id2idx[s_id]
        e_idx = id2idx[e_id]
        if s_idx > e_idx:
            s_idx, e_idx = e_idx, s_idx

        text = words_span_text(word_times, s_idx, e_idx)
        tok_len = count_llama_tokens(text)

        utterances.append({
            "segment_id": seg["segment_id"],
            "start_time": seg["start_time"],
            "end_time": seg["end_time"],
            "start_word_idx": s_idx,
            "end_word_idx": e_idx,
            "token_len": tok_len,
            "text": text,
        })

    avg_tok = sum(u["token_len"] for u in utterances) / len(utterances) if utterances else 0.0
    return utterances, avg_tok


# ---------------------------
# 6) Topic parsing (topic.xml)
# ---------------------------
def parse_topics(topic_path, word_times, id2idx):
    tree = etree.parse(str(topic_path))
    root = tree.getroot()

    topics = []
    for topic in root.findall('.//topic'):
        label = topic.get('other_description', 'Unknown').strip()

        start_idx_list = []
        end_idx_list = []
        start_time_list = []
        end_time_list = []

        childs = topic.findall('.//nite:child', namespaces=NS)
        for child in childs:
            href = child.get('href')
            if not href or "#" not in href:
                continue

            _, fragment = href.split('#', 1)
            parts = fragment.replace('id(', '').replace(')', '').split('..')
            start_id = parts[0]
            end_id = parts[1] if len(parts) == 2 else start_id

            if start_id not in id2idx or end_id not in id2idx:
                continue

            s_idx = id2idx[start_id]
            e_idx = id2idx[end_id]
            if s_idx > e_idx:
                s_idx, e_idx = e_idx, s_idx

            start_idx_list.append(s_idx)
            end_idx_list.append(e_idx)
            start_time_list.append(word_times[s_idx]["start"])
            end_time_list.append(word_times[e_idx]["end"])

        if not start_idx_list:
            continue

        topics.append({
            "label": label,
            "start_word_idx": min(start_idx_list),
            "end_word_idx": max(end_idx_list),
            "start_time": min(start_time_list),
            "end_time": max(end_time_list),
        })

    return topics


def topic_token_stats(topics_ref, word_times):
    per_topic = []
    for t in topics_ref:
        s_idx = t["start_word_idx"]
        e_idx = t["end_word_idx"]
        text = words_span_text(word_times, s_idx, e_idx)
        tok_len = count_llama_tokens(text)

        per_topic.append({
            "label": t["label"],
            "start_time": t["start_time"],
            "end_time": t["end_time"],
            "start_word_idx": s_idx,
            "end_word_idx": e_idx,
            "token_len": tok_len,
            "text": text,
        })

    avg_tok = sum(x["token_len"] for x in per_topic) / len(per_topic) if per_topic else 0.0
    return per_topic, avg_tok

def run_meeting(meeting_id: str):
    word_files = glob.glob(str(WORDS_DIR / f"{meeting_id}.*.words.xml"))
    seg_files = glob.glob(str(SEGMENTS_DIR / f"{meeting_id}.*.segments.xml"))
    topic_xml = TOPICS_DIR / f"{meeting_id}.topic.xml"

    if not word_files:
        raise FileNotFoundError(f"No words.xml for {meeting_id}")
    if not seg_files:
        raise FileNotFoundError(f"No segments.xml for {meeting_id}")
    if not topic_xml.exists():
        raise FileNotFoundError(f"No topic.xml for {meeting_id}")

    word_times, _, id2idx = combine_words_from_multiple_files(word_files)

    # utterances
    segments = combine_segments_from_multiple_files(seg_files)
    utts, utt_avg = utterance_token_stats(segments, word_times, id2idx)

    # topics
    topics_ref = parse_topics(topic_xml, word_times, id2idx)
    tops, top_avg = topic_token_stats(topics_ref, word_times)

    out = {
        "meeting_id": meeting_id,
        "avg_utterance_token_len": utt_avg,
        "avg_topic_token_len": top_avg,
        "utterances": utts,
        "topics": tops,
    }

    # out_path = OUT_DIR / f"{meeting_id}.llama_token_lengths.json"
    # with open(out_path, "w", encoding="utf-8") as f:
    #     json.dump(out, f, indent=2, ensure_ascii=False)

    print(f"[{meeting_id}] utterances={len(utts)} avg_utt_tok={utt_avg:.2f} | topics={len(tops)} avg_topic_tok={top_avg:.2f}")
    # print("Saved:", out_path)
    return out



In [15]:
def run_corpus():
    total_utt_tok = 0
    total_utt_cnt = 0
    total_topic_tok = 0
    total_topic_cnt = 0
    total_topic_dur = 0
    used_files = os.listdir("/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/amicorpus")

    for tf in TOPICS_DIR.glob("*.topic.xml"):
        if tf.name.split(".")[0] not in used_files:
            continue

        meeting_id = tf.stem.split(".")[0]
        print("Processing meeting:", meeting_id)
        try:
            out = run_meeting(meeting_id)
            print(out)
        except FileNotFoundError:
            continue

        total_utt_tok += sum(u["token_len"] for u in out["utterances"])
        total_utt_cnt += len(out["utterances"])
        total_topic_tok += sum(t["token_len"] for t in out["topics"])
        total_topic_cnt += len(out["topics"])
        total_topic_dur += sum(t["end_time"] - t["start_time"] for t in out["topics"])

    corpus = {
        "avg_utterance_token_len": (total_utt_tok / total_utt_cnt) if total_utt_cnt else 0.0,
        "avg_topic_token_len": (total_topic_tok / total_topic_cnt) if total_topic_cnt else 0.0,
        "avg_topic_duration": (total_topic_dur / total_topic_cnt) if total_topic_cnt else 0.0,
        "utterance_count": total_utt_cnt,
        "topic_count": total_topic_cnt,
        "total_topic_duration": total_topic_dur,
    }

    out_path = "/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/topic_stats/ami_corpus_llama_token_summary.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(corpus, f, indent=2, ensure_ascii=False)

    print("[AMI CORPUS]", corpus)
    print("Saved:", out_path)

run_corpus()

Processing meeting: ES2013a
[ES2013a] utterances=156 avg_utt_tok=14.77 | topics=10 avg_topic_tok=265.20
{'meeting_id': 'ES2013a', 'avg_utterance_token_len': 14.76923076923077, 'avg_topic_token_len': 265.2, 'utterances': [{'segment_id': None, 'start_time': 22.483, 'end_time': 23.686, 'start_word_idx': 0, 'end_word_idx': 3, 'token_len': 4, 'text': 'Is this okay?'}, {'segment_id': None, 'start_time': 27.392, 'end_time': 32.224, 'start_word_idx': 4, 'end_word_idx': 9, 'token_len': 6, 'text': 'Uh yeah. Fine now.'}, {'segment_id': None, 'start_time': 47.36, 'end_time': 51.056, 'start_word_idx': 10, 'end_word_idx': 20, 'token_len': 14, 'text': "Oh, it's not liking us, it went that-a-way."}, {'segment_id': None, 'start_time': 53.943, 'end_time': 56.786, 'start_word_idx': 21, 'end_word_idx': 25, 'token_len': 5, 'text': 'Computer adjusting. Oh.'}, {'segment_id': None, 'start_time': 61.968, 'end_time': 62.32, 'start_word_idx': 26, 'end_word_idx': 27, 'token_len': 2, 'text': 'Uh.'}, {'segment_id':

In [13]:
import os
import json

DURANTION_FILE_ROOT = "/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json"

TOTAL_DURATION_UTTEREANCE = 0
ALL_UTTERANCE_COUNT = 0


for file in os.listdir(DURANTION_FILE_ROOT):
    if file.endswith(".json"):
        with open(os.path.join(DURANTION_FILE_ROOT, file), 'r') as f:
            data = json.load(f)
            for key,segment in data.items():
                start_time = segment['start_time']
                end_time = segment['end_time']
                TOTAL_DURATION_UTTEREANCE += (end_time - start_time)
                ALL_UTTERANCE_COUNT += 1


avg_duration_per_utterance = TOTAL_DURATION_UTTEREANCE / ALL_UTTERANCE_COUNT if ALL_UTTERANCE_COUNT > 0 else 0
print(f"Total Duration of all Utterances (seconds): {TOTAL_DURATION_UTTEREANCE:.2f}")
print(f"Total Utterances for Duration Calculation: {ALL_UTTERANCE_COUNT}")
print(f"Average Duration per Utterance (seconds): {avg_duration_per_utterance:.2f}")    

Total Duration of all Utterances (seconds): 42285.74
Total Utterances for Duration Calculation: 9058
Average Duration per Utterance (seconds): 4.67
